In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import json

In [7]:
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA AVAILABILITY INTERACTIVE DASHBOARD'
datafile='final_dataset_combined.xlsx'

full_path = os.path.join(path, datafile)

# Read the Excel file
data = pd.read_excel(full_path)

### Translate

In [17]:
#read in the dictionary
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
df_translation=pd.read_excel(path+'/translation dict.xlsx')

In [18]:
df_translation.head()

,col_en,val_en,col_ar,val_ar
0,Age Group,0-1 years,الفئة العمرية,0-1 سنة
1,Age Group,0-4 years,الفئة العمرية,0-4 سنوات
2,Age Group,5-17 years,الفئة العمرية,5-17 سنة
3,Age Group,10-14 years,الفئة العمرية,10-14 سنة
4,Age Group,1-4 years,الفئة العمرية,1-4 سنوات


In [ ]:
En_Ar_dictionary={}
#get the unique dimensions
# dimensions = set(df_translation['col_en'].unique()) - {'Year', 'Value', 'Source'}
dimensions = df_translation['col_en'].unique()

for dim in dimensions:
    df_dim=df_translation[df_translation['col_en'].isin([dim.lower(), dim])].copy()
    En_Ar_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_en'], df_dim['val_ar'])), 
                'dim': {df_dim['col_en'].unique()[0]:df_dim['col_ar'].unique()[0]}}})

Ar_En_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_ar'].unique()) - {'السنة', 'العدد', 'المصدر'}

for dim in dimensions:
    df_dim=df_translation[df_translation['col_ar'].isin([dim.lower(), dim])].copy()
    Ar_En_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_ar'], df_dim['val_en'])), 
                'dim': {df_dim['col_ar'].unique()[0]:df_dim['col_en'].unique()[0]}}})
    
###################################################################################################
    
class Translator:
    def __init__(self, translate_to, En_Ar_dictionary, Ar_En_dictionary):
        self.translate_to = translate_to.lower()
        self.en_ar_dict = En_Ar_dictionary
        self.ar_en_dict = Ar_En_dictionary

    def translate(self, df):
        df_translated = df.copy()
        
        # Pick the dictionary based on direction
        mapping = self.ar_en_dict if self.translate_to == 'english' else self.en_ar_dict
        
        for col, col_dict in mapping.items():
            if col in df_translated.columns:
                #Replace the data values (e.g., 'Male' -> 'ذكر')
                df_translated[col] = df_translated[col].replace(col_dict['dim_values'])
                
                #Rename the column header (e.g., 'Gender' -> 'الجنس')
                df_translated.rename(columns=col_dict['dim'], inplace=True)
                
        return df_translated

In [21]:
translator = Translator('english',En_Ar_dictionary, Ar_En_dictionary)
en_df_translated = translator.translate(data)

In [23]:
en_df_translated.columns

Index(['Indicator', 'Country', 'الفصل', 'السنة', 'العدد', 'المصدر',
       'Nationality', 'Sex', 'Education level', 'Age Group', 'Area',
       'Source of Lighting', 'Types of sewage disposal system',
       'Tenure of housing unit', 'Source of water supply',
       'Type of living quarter', 'Sector', 'Main occupation',
       'Economic activity', 'Employment status', 'Reasons for inactivity',
       'International classification for causes of death', 'Causes of death',
       'Marital status', 'Quintile', 'Types of products/services', 'Sector'],
      dtype='object')

In [24]:
df_translation['col_en'].unique()

array(['Age Group', 'Area', 'Causes of death',
       'International classification for causes of death',
       'Marital status', 'Country', 'Economic activity',
       'Education level', 'Employment status', 'Indicator',
       'Main occupation', 'Nationality', 'Quintile',
       'Reasons for inactivity', 'Sector', 'Sex', 'Source',
       'Source of Lighting', 'Source of water supply',
       'Tenure of housing unit', 'Type of living quarter',
       'Types of products/services', 'Types of sewage disposal system',
       'Value', 'Year'], dtype=object)

In [ ]:
# 1. Prepare your data
# Example: 15 countries, each with 20 data points
countries = [
    "Algeria", "Bahrain", "Egypt", "Iraq", "Jordan", "Kuwait", "Lebanon", 
    "Libya", "Mauritania", "Morocco", "Oman", "State of Palestine", "Qatar", 
    "Saudi Arabia", "Sudan", "Syrian Arab Republic", "Tunisia", "United Arab Emirates", "Yemen"
]

# Create a figure with a grid of subplots (e.g., 5 rows, 4 columns)
# Adjust figsize as needed to fit your aspect ratio
fig, axes = plt.subplots(5, 4, figsize=(15, 12), sharex=True, sharey=True)

# Flatten the axes array to iterate over it easily
axes_flat = axes.flatten()

# 2. Iterate and plot
for i, ax in enumerate(axes_flat):
    if i < len(countries):
        # Generate dummy data for each country
        x = np.arange(20)
        y = 40 + np.random.rand(20) * 30  # Random values between 40 and 70
        
        # Plot the data
        ax.plot(x, y, marker='o', linestyle='-', markersize=4)
        
        # Set the title for each subplot
        ax.set_title(countries[i], fontsize=10)
        
        # Hide standard tick labels to match your screenshot style
        ax.set_xticks([])
        ax.set_yticks([40, 60, 80])
    else:
        # Hide unused subplots if the number of countries isn't a multiple of the grid
        ax.axis('off')

# 3. Final adjustments
plt.tight_layout()

# 4. Save as SVG
plt.savefig("country_charts.svg", format="svg")
plt.show()